<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/leaderboard_focus/01_CHAMPION_RECOVERY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M1 · 01 — Champion recovery (CPU)

**Purpose.** Identify the strongest **valid, deployable** existing W2 configuration from the frozen boost-A grid with the predeclared p6g selection rule. No new model, no grid recomputation.

| | |
|---|---|
| Runtime | CPU, High-RAM recommended (unit crops of the full grid) |
| Prerequisites | notebook 00 status COMPLETE |
| Inputs | `p6_fast_track/boost/benchmark/grid.csv`, `stack/maps/*.npy`, `stack/candidate.json`, `recovery/export_parity/*`, B0 / B1 OOF maps, M9 baseline selection |
| Outputs | `selection/{champion_candidates.csv, champion_selection.json, champion_recipe.yaml, champion_provenance.json, blocked_candidates.csv, candidate_per_zone.csv, protocol_consistency.csv}`, `reports/recovery_report.md` |
| Next | `02_OFFICIAL_VALIDATION.ipynb` when the selection status is COMPLETE |

**Selection rule (predeclared, `configs/p6/full_gpu.yaml` → `selection`).**
* For each outer zone, ONE cell is chosen on the other zones and applied to that zone in both tasks.
* The objective is `disc`.
* Eligibility requires a worst zone ≥ 0.153414 and the relative geographic rule.
* A replacement must gain ≥ 0.005 and be no worse on ≥ 2 of 3 zones.

The 0.1758 geographic floor is reported per candidate (`geo_floor_met`). It blocks only if you set `M1_CONFIG = {'selection': {'geo_floor_blocking': True}}`.

## 1 · Configuration

In [1]:
CONFIG = {'repo': 'mybenkhadda/GEMS---drivendata', 'ref': 'main', 'drive_root': '/content/drive/MyDrive/GEMS_2',
          'local_root': '/content/gems_work', 'reuse_cached': True, 'strict_provenance': True,
          'overwrite_existing': False, 'device': 'cpu'}
PRIVATE_REPO = False
M1_CONFIG = {}
EXPECT = {'code_rev': 'm1.1-r1', 'commit': None}

## 2 · Drive, strict clone, version lock, shared session

`M1Session.attach` stages cached data (never rebuilt), builds M9 → P6 → p6g in `mode: recovery` with its output root = the M1 root, and refuses to start without a COMPLETE notebook-00 inventory.

In [2]:
import base64, os, subprocess, sys, json
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
def git(*a, check=True):
    env = None
    if PRIVATE_REPO:
        from google.colab import userdata
        hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
        env = {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}
    r = subprocess.run(['git', *a], capture_output=True, text=True, env=env)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(a[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r
if not Path(CODE, '.git').exists():
    git('clone', '-q', 'https://github.com/' + CONFIG['repo'] + '.git', CODE)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin')
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + CONFIG['ref'], check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'm1_colab_active', 'origin/' + CONFIG['ref'])
else:
    git('-C', CODE, 'checkout', '-q', '--detach', CONFIG['ref'])
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: restart the runtime')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
import gems_bench.leaderboard_focus as LF
assert LF.M1_CODE_REV == EXPECT['code_rev'] and (not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']))
from gems_bench.leaderboard_focus.session import M1Session
m1 = M1Session.attach(CONFIG, M1_CONFIG)
print('HEAD', HEAD, '| M1 root', m1.ws.root, '| staged W2 inputs', m1.staged)

Mounted at /content/drive
[sync out] 2 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/config_snapshot.json', 'reports/pipeline/environment_report.csv']
[sync out] 1 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/stages/bootstrap.json']
[bootstrap] DONE: bootstrap: ok (3 checks)
[bootstrap] outputs: ['/content/drive/MyDrive/GEMS_2/environment/pipeline/config_snapshot.json', '/content/drive/MyDrive/GEMS_2/reports/pipeline/environment_report.csv']
[sync out] 2 copied (0.0 MiB), 0 unchanged, 0s: ['data/raw/acquisition_manifest.json', 'reports/pipeline/raw_data_acquisition.csv']
[sync out] 1 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/stages/raw_data.json']
[raw_data] DONE: raw_data: ok (3 checks)
[raw_data] outputs: ['/content/drive/MyDrive/GEMS_2/data/raw/acquisition_manifest.json', '/content/drive/MyDrive/GEMS_2/reports/pipeline/raw_data_acquisition.csv']
[sync in] 1 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/native_grid.json']
[native_g

## 3 · Recover the champion (reloads while the fingerprint is unchanged)

In [4]:
from pathlib import Path
import pandas as pd

root = m1.fg.p6.ws.root

for name, spec in m1.fg.cfg["grids"].items():
    path = root / spec["path"]

    if not path.exists():
        print(f"{name:12} MISSING     {path}")
        continue

    size = path.stat().st_size

    try:
        df = pd.read_csv(path, nrows=3)
        status = f"VALID ({len(df.columns)} columns)"
    except pd.errors.EmptyDataError:
        status = "EMPTY CSV"
    except Exception as e:
        status = f"ERROR: {type(e).__name__}: {e}"

    print(f"{name:12} {size:>10,} bytes  {status}")

boost_a          99,538 bytes  VALID (28 columns)
boost_b      MISSING     /content/gems_work/outputs/p6_fast_track/boost/geology/grid.csv
boost_c      MISSING     /content/gems_work/outputs/p6_fast_track/boost/robustness/grid.csv
boost_d      MISSING     /content/gems_work/outputs/p6_fast_track/boost/hybrid/grid.csv
rec_x3           58,416 bytes  VALID (28 columns)
w1_seeds              1 bytes  EMPTY CSV


In [5]:
# Temporary Milestone 1 workaround:
# Only load the existing Boost A benchmark.
from copy import deepcopy

m1.fg.cfg = deepcopy(m1.fg.cfg)
m1.fg.cfg["grids"] = {
    "boost_a": m1.fg.cfg["grids"]["boost_a"]
}

from gems_bench.leaderboard_focus.champion import recover

M = recover(m1)

print("Champion recovery status:", M["status"])

[sync in] 1 copied (0.0 MiB), 0 unchanged, 0s: ['outputs/m9_geofault_production/baseline/baseline_selection.json']
[sync out] 10 copied (0.0 MiB), 5 unchanged, 0s: ['outputs/leaderboard_focus/m1_champion_recovery']
Champion recovery status: COMPLETE


In [6]:
from IPython.display import Markdown, display
from gems_bench.leaderboard_focus.champion import recover
M = recover(m1)
SEL = m1.ws.read_json('selection/champion_selection.json') or {}
print('SELECTION STATUS:', M['status'], '| champion:', SEL.get('champion'), '| decision:', SEL.get('decision'), '| recipe:', SEL.get('recipe_id'))
for r in M['reasons']:
    print('  reason:', r)
C = m1.ws.load_table('selection/champion_candidates.csv')
display(C[['candidate_id', 'role', 'status', 'eligible', 'disc_dti', 'disc_worst', 'geo_dti', 'geo_floor_met', 'recorded_disc_dti', 'decision']])

[m1] selection: fingerprint fce3311abb unchanged -> reloaded (status COMPLETE)
SELECTION STATUS: COMPLETE | champion: a_w2_rep_frac | decision: REPLACE | recipe: 0a8692b2ab


,candidate_id,role,status,eligible,disc_dti,disc_worst,geo_dti,geo_floor_met,recorded_disc_dti,decision
0,ref_p6_w2,reference,EVALUATED,True,0.224640,0.153414,0.102664,False,NaN,BASELINE
1,a_w2_crest,candidate,EVALUATED,True,0.238882,0.192575,0.114259,False,0.238882,ELIGIBLE_NOT_CHOSEN
2,a_w2_skeleton,candidate,EVALUATED,True,0.242156,0.198579,0.105687,False,0.242156,ELIGIBLE_NOT_CHOSEN
3,a_w2_rep_frac,candidate,EVALUATED,True,0.242156,0.198579,0.105687,False,NaN,CHAMPION
4,a_hybrid,candidate,EVALUATED,False,0.228417,0.153414,0.126658,False,NaN,INELIGIBLE
5,rec_p6r_choice,recovery_baseline,EVALUATED,True,0.242156,0.198579,0.105687,False,NaN,ELIGIBLE_NOT_CHOSEN
6,ref_m9_fallback,reference,EVALUATED,False,0.162482,0.126160,0.158416,False,NaN,INELIGIBLE
7,m1_w2_uploaded_cell,historical,EVALUATED,False,0.178905,0.155761,0.168841,False,0.178905,INELIGIBLE


## 4 · Verification checklist

1. `SELECTION STATUS: COMPLETE` with a champion and a recipe id.
2. `selection/protocol_consistency.csv`: every `ok` True.
3. `blocked_candidates.csv`: every blocked candidate has a reason (missing maps, worst-zone floor, historical role, …).
4. `recorded_*` columns are historical records; `disc_dti` / `geo_dti` are the recomputed nested single-policy numbers. They differ by construction when the record came from separate disc / geo selections.

## 5 · Troubleshooting

* `StageBlocked: artifact_inventory.json ...`: run notebook 00.
* `NO_ELIGIBLE_CANDIDATE`: no candidate passes the predeclared rule. M1 stops (BLOCKED) instead of exporting the reference silently; inspect `eligibility_reasons`.
* Memory error: switch to a High-RAM runtime (each unit crop of the B0 / B1 / W2 maps is held in RAM once).

## 6 · Next

`02_OFFICIAL_VALIDATION.ipynb`.

In [7]:
display(Markdown(m1.ws.path('reports/recovery_report.md').read_text(encoding='utf-8')))

# M1 champion recovery

Status: **COMPLETE** | champion `a_w2_rep_frac` | decision REPLACE | objective `disc` | recipe `0a8692b2ab`

Numbers: nested single-policy estimates from the frozen boost-A grid (one cell per outer zone, chosen on the other zones). `recorded_*` columns are historical records from the configs.

| candidate_id        | role              | status    | eligible   |   disc_dti |   disc_worst |   geo_dti | geo_floor_met   |   recorded_disc_dti |   recorded_geo_dti | decision            | eligibility_reasons                                                                    |
|:--------------------|:------------------|:----------|:-----------|-----------:|-------------:|----------:|:----------------|--------------------:|-------------------:|:--------------------|:---------------------------------------------------------------------------------------|
| ref_p6_w2           | reference         | EVALUATED | True       |   0.22464  |     0.153414 |  0.102664 | False           |          nan        |         nan        | BASELINE            |                                                                                        |
| a_w2_crest          | candidate         | EVALUATED | True       |   0.238882 |     0.192575 |  0.114259 | False           |            0.238882 |           0.185609 | ELIGIBLE_NOT_CHOSEN |                                                                                        |
| a_w2_skeleton       | candidate         | EVALUATED | True       |   0.242156 |     0.198579 |  0.105687 | False           |            0.242156 |           0.168841 | ELIGIBLE_NOT_CHOSEN |                                                                                        |
| a_w2_rep_frac       | candidate         | EVALUATED | True       |   0.242156 |     0.198579 |  0.105687 | False           |          nan        |         nan        | CHAMPION            |                                                                                        |
| a_hybrid            | candidate         | EVALUATED | False      |   0.228417 |     0.153414 |  0.126658 | False           |          nan        |         nan        | INELIGIBLE          | ensemble gain -0.0137 over best member a_w2_rep_frac < 0.005                           |
| rec_p6r_choice      | recovery_baseline | EVALUATED | True       |   0.242156 |     0.198579 |  0.105687 | False           |          nan        |         nan        | ELIGIBLE_NOT_CHOSEN |                                                                                        |
| ref_m9_fallback     | reference         | EVALUATED | False      |   0.162482 |     0.12616  |  0.158416 | False           |          nan        |         nan        | INELIGIBLE          | worst zone 0.1262 < 0.153414; ensemble members not evaluated: ['a_cnn_b0', 'a_cnn_b1'] |
| m1_w2_uploaded_cell | historical        | EVALUATED | False      |   0.178905 |     0.155761 |  0.168841 | False           |            0.178905 |           0.168841 | INELIGIBLE          | role historical is never selected                                                      |

## Reasons

* none

META (honest estimate of choosing among eligible candidates): {'status': 'RUN', 'disc_dti': 0.22464028846915302, 'disc_worst': 0.1534143947207408, 'candidates': ['ref_p6_w2', 'a_w2_crest', 'a_w2_skeleton', 'a_w2_rep_frac', 'rec_p6r_choice']}
